In [ ]:
# Cell 1: Import necessary libraries and environment setup
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Wav2Vec2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, accuracy_score

# Configure device and check GPU specifications
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("-" * 50)
print(f"🚀 Execution Device: {device}")

if torch.cuda.is_available():
    # Let's flex the hardware for your teammates!
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3) # Convert to GB
    print(f"🔥 Hardware Model: {gpu_name}")
    print(f"📟 Dedicated VRAM: {gpu_memory:.2f} GB")
    print(f"✨ CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Warning: Running on CPU. Performance might be limited.")
print("-" * 50)

# Global setting: Optimization for large-scale audio processing
# Note: We are using librosa for loading to bypass torchaudio/torchcodec version conflicts
import librosa 
print("✅ Environment initialized. Ready for FinQuest 2026.")

In [ ]:
# Cell 2: Load Master Protocol and Perform Dataset Splitting (Train/Dev/Test)
# We partition the data using an 80/10/10 ratio for Training, Development, and Testing.
protocol_path = "./protocol/master_protocol.csv"
df = pd.read_csv(protocol_path)

# First split: Separate 80% for Training and 20% for the remaining pool (Temp)
# Note: 'stratify' ensures the Real/Fake ratio remains consistent across all sets.
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

# Second split: Evenly divide the remaining 20% into Dev and Test sets (10% each of total)
dev_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'], 
    random_state=42
)

print("-" * 60)
print(f"📊 Dataset Statistics:")
print(f"Total Samples: {len(df)}")
print(f"Training Set : {len(train_df)} samples (80%)")
print(f"Validation Set: {len(dev_df)} samples (10%)")
print(f"Testing Set   : {len(test_df)} samples (10%)")
print("-" * 60)

# Verification of class balance for the team
print("✅ Stratification Check (Real vs Fake ratio):")
print(f"Train Label 1 Ratio: {train_df['label'].mean():.2%}")
print(f"Dev   Label 1 Ratio: {dev_df['label'].mean():.2%}")
print(f"Test  Label 1 Ratio: {test_df['label'].mean():.2%}")
print("-" * 60)

In [ ]:
# Cell 3: Professional Augmented Dataset (Fully Fixed for Audiomentations 2026)
import librosa
import torch
import numpy as np
import random
from torch.utils.data import Dataset
from audiomentations import Compose, AddGaussianNoise, Shift, Gain, HighPassFilter, SevenBandParametricEQ

class FinQuestDataset(Dataset):
    """
    Professional Dataset with latest Audiomentations API compatibility.
    """
    def __init__(self, dataframe, processor, max_length=64000, augment=False):
        self.df = dataframe
        self.processor = processor
        self.max_length = max_length
        self.augment = augment
        
        if self.augment:
            self.augmenter = Compose([
                # 1. 模拟音量波动
                Gain(min_gain_db=-10, max_gain_db=5, p=0.5),
                
                # 2. 模拟麦克风特征 (对付翻录)
                HighPassFilter(min_cutoff_freq=100, max_cutoff_freq=500, p=0.3),
                SevenBandParametricEQ(min_gain_db=-6, max_gain_db=6, p=0.4),
                
                # 3. 基础环境噪声
                AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.3),
                
                # 4. 🚀 关键修正：参数名由 min_fraction 改为 min_shift
                Shift(min_shift=-0.5, max_shift=0.5, p=0.5),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        label = row['label']

        try:
            # 加载音频
            waveform, _ = librosa.load(path, sr=16000, mono=True)
            
            # 数据增强
            if self.augment:
                # 确保是 float32 格式
                waveform = self.augmenter(samples=waveform.astype(np.float32), sample_rate=16000)
            
            # 随机裁剪 (4s)
            num_samples = len(waveform)
            if num_samples > self.max_length:
                if self.augment:
                    start = random.randint(0, num_samples - self.max_length)
                    waveform = waveform[start : start + self.max_length]
                else:
                    start = (num_samples - self.max_length) // 2
                    waveform = waveform[start : start + self.max_length]
            else:
                padding_needed = self.max_length - num_samples
                waveform = np.pad(waveform, (0, padding_needed))

            # 转换为 Tensor 并提取特征
            waveform_t = torch.from_numpy(waveform.copy())
            inputs = self.processor(waveform_t, sampling_rate=16000, return_tensors="pt")
            
            return {
                "input_values": inputs.input_values.squeeze(0),
                "label": torch.tensor(label, dtype=torch.long)
            }
        except Exception as e:
            return self.__getitem__((idx + 1) % len(self.df))

print("✅ Cell 3: Dataset class defined with 2026 API standards.")

In [ ]:
# Cell 4: Model Architecture Definition (Wav2Vec2 + Attentive Pooling + MLP)

class AttentivePooling(torch.nn.Module):
    """
    Self-Attentive Pooling layer to aggregate frame-level features into a 
    single utterance-level representation by focusing on relevant temporal segments.
    """
    def __init__(self, hidden_size):
        super().__init__()
        # Attention mechanism to compute importance scores for each frame
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )

    def forward(self, x):
        # Input shape x: (batch, seq_len, hidden_size)
        # Compute softmax-normalized attention weights across the time dimension
        attn_weights = torch.softmax(self.attention(x), dim=1)
        # Weighted summation of hidden states
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

# Step 1: Load Pre-trained Wav2Vec 2.0 Base Engine
# This model provides robust acoustic feature extraction from raw audio.
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)

# Step 2: Initialize Classification Head
# 768 is the hidden size for the wav2vec2-base transformer layers.
pooling = AttentivePooling(768).to(device)

# Multi-Layer Perceptron (MLP) for final binary classification (Real vs Fake)
classifier = torch.nn.Sequential(
    torch.nn.Linear(768, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),  # Regularization to prevent overfitting
    torch.nn.Linear(256, 2)
).to(device)

print(f"✅ Model architecture initialized on {device}.")
print(f"🔹 Backbone: Wav2Vec2-Base (Frozen/Fine-tuned)")
print(f"🔹 Aggregation: Self-Attentive Pooling")
print(f"🔹 Output: Binary Logits [Fake, Real]")

In [ ]:
# Cell 5: Training Hyperparameters & Optimizer Configuration (Enhanced Version)
from torch.utils.data import DataLoader

# 1. Initialize Datasets
# 🚀 激活 audiomentations：训练集 augment=True，验证集 augment=False
train_dataset = FinQuestDataset(train_df, processor, augment=True)
dev_dataset   = FinQuestDataset(dev_df, processor, augment=False)

# 2. DataLoaders
# RTX 5080 算力强大，batch_size=16 能提供更稳定的梯度，num_workers=4 加速 CPU 增强处理
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
dev_loader   = DataLoader(dev_dataset, batch_size=16, num_workers=4)

# 3. Layer-wise Learning Rate Decay (LLRD) Strategy
# 针对 Wav2Vec2 的层级微调策略，保护预训练特征，重点训练分类头
optimizer = torch.optim.AdamW([
    # 第一梯队：卷积特征提取器 (保持极低 LR)
    {"params": backbone.feature_extractor.parameters(), "lr": 2e-6}, 
    
    # 第二梯队：Transformer 编码器 (中等 LR)
    {"params": backbone.encoder.parameters(), "lr": 8e-6},
    
    # 第三梯队：Attentive Pooling & Classifier 分类头 (高 LR，全力学习新模式)
    {"params": pooling.parameters(), "lr": 8e-5},
    {"params": classifier.parameters(), "lr": 8e-5},
], weight_decay=0.01)

# 4. Standard Loss Function for Binary Classification
criterion = torch.nn.CrossEntropyLoss()

# 5. Training Summary Report
print("-" * 60)
print("🚀 FINQUEST 2026: ADVANCED TRAINING CONFIG")
print("-" * 60)
print(f"🔹 Dataset Mode  : Audiomentations (Random Crop 4s)")
print(f"🔹 Augmentation  : ENABLED (Training Only)")
print(f"🔹 Batch Size    : {train_loader.batch_size}")
print(f"🔹 Optimizer     : AdamW with LLRD")
print(f"🔹 CNN/Enc/Head  : 2e-6 / 8e-6 / 8e-5")
print(f"🔹 Device        : RTX 5080 (Optimization Active)")
print("-" * 60)
print("✅ Ready to forge a robust deepfake detector.")

In [ ]:
# Cell 6: Training & Validation Loop (Warmup & LLRD Optimized Edition)
import os
import warnings
import logging
import torch
import numpy as np
import csv
from tqdm import tqdm
from sklearn.metrics import roc_curve, accuracy_score
# --- 新增：引入 Transformers 的 Scheduler ---
from transformers import get_cosine_schedule_with_warmup

# ==============================================================================
# 🚀 ULTIMATE NOISE SUPPRESSION
# ==============================================================================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore") 
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger('librosa').setLevel(logging.ERROR)
logging.getLogger('audioread').setLevel(logging.ERROR)

# Ensure the model and log storage directory exists
MODEL_DIR = "./model"
LOG_FILE = os.path.join(MODEL_DIR, "training_log.csv")
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Initialize CSV Log File ---
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "dev_acc", "dev_eer", "is_best"])

def save_metrics_to_csv(epoch, loss, acc, eer, is_best):
    with open(LOG_FILE, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{loss:.6f}", f"{acc:.4f}", f"{eer:.4f}", "YES" if is_best else "NO"])

def evaluate(model_base, pool_layer, clf_layer, loader):
    model_base.eval()
    pool_layer.eval()
    clf_layer.eval()
    all_scores, all_labels = [], []
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with torch.no_grad():
            for batch in tqdm(loader, desc="Validating", leave=False):
                inputs = batch["input_values"].to(device)
                labels = batch["label"].to(device)
                
                outputs = model_base(inputs).last_hidden_state
                pooled = pool_layer(outputs)
                logits = clf_layer(pooled)
                
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_scores.extend(probs)
                all_labels.extend(labels.cpu().numpy())
            
    all_labels = np.array(all_labels)
    all_scores = np.array(all_scores)
    
    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=1)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
    acc = accuracy_score(all_labels, all_scores > 0.5)
    return acc, eer

# --- 🚀 新增：Warmup Scheduler 配置 ---
max_epochs = 20    
patience = 3       
num_training_steps = len(train_loader) * max_epochs
# 预热步数设为总步数的 10%，让模型在 BS=16 的大步长下平稳起步
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=num_warmup_steps, 
    num_training_steps=num_training_steps
)

# --- Training Configuration & Early Stopping ---
best_eer = 1.0     
counter = 0        

print(f"🏁 FinQuest 2026: Training initiated. BS={train_loader.batch_size}, Warmup={num_warmup_steps} steps.")
print(f"📈 Metrics will be saved to: {LOG_FILE}")

for epoch in range(max_epochs):
    backbone.train()
    pooling.train()
    classifier.train()
    total_loss = 0
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for batch in pbar:
            optimizer.zero_grad()
            
            inputs = batch["input_values"].to(device)
            labels = batch["label"].to(device)
            
            outputs = backbone(inputs).last_hidden_state
            pooled = pooling(outputs)
            logits = classifier(pooled)
            
            loss = criterion(logits, labels)
            loss.backward()
            
            # --- 🚀 关键修改：更新参数的同时更新学习率 ---
            optimizer.step()
            scheduler.step() 
            
            total_loss += loss.item()
            # 在 tqdm 中显示当前最后一位参数组（即 Classifier）的学习率，方便监控
            current_lr = optimizer.param_groups[-1]['lr']
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.2e}")

    # --- Validation Phase ---
    dev_acc, dev_eer = evaluate(backbone, pooling, classifier, dev_loader)
    avg_loss = total_loss / len(train_loader)
    
    is_best = dev_eer < best_eer
    save_metrics_to_csv(epoch, avg_loss, dev_acc, dev_eer, is_best)
    
    print(f"\n📊 [Epoch {epoch} Summary]")
    print(f"   - Training Loss : {avg_loss:.6f}")
    print(f"   - Dev Accuracy  : {dev_acc:.4f}")
    print(f"   - Dev EER       : {dev_eer:.4f}")
    
    if is_best:
        best_eer = dev_eer
        counter = 0  
        save_path = os.path.join(MODEL_DIR, "best_deepfake_model.pth")
        
        clean_state_dict = {
            'backbone': backbone.state_dict(),
            'pooling': pooling.state_dict(),
            'classifier': classifier.state_dict()
        }
        
        torch.save(clean_state_dict, save_path)
        print(f"🥇 New Best Model! Performance improved. Pure weights saved to {save_path}")
    else:
        counter += 1
        print(f"⏱ No improvement. Counter: {counter}/{patience}")
    
    if counter >= patience:
        print(f"🛑 Early Stopping triggered. Best EER: {best_eer:.4f}")
        break
    print("-" * 50)

print("\n✨ Training session complete. All logs are in CSV format.")

In [ ]:
# Cell 7: Full Test Evaluation and Model Source Attribution Report
import librosa
from sklearn.metrics import roc_auc_score, confusion_matrix

# 1. Load the optimal weights from the model directory
print("🔄 Loading best model weights for final evaluation...")
checkpoint = torch.load("model/best_deepfake_model.pth", weights_only=False)
backbone.load_state_dict(checkpoint['backbone'])
pooling.load_state_dict(checkpoint['pooling'])
classifier.load_state_dict(checkpoint['classifier'])

backbone.eval()
pooling.eval()
classifier.eval()

all_scores = []
all_labels = []
source_results = {} # Metrics storage for cross-algorithm traceability

print(f"🔍 Running full inference on {len(test_df)} unseen test samples...")

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference Phase"):
        try:
            # Step 1: Pre-processing
            wav, _ = librosa.load(row['path'], sr=16000, mono=True)
            inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
            
            # Step 2: Forward pass through the inference pipeline
            out = backbone(inputs).last_hidden_state
            pooled = pooling(out)
            logits = classifier(pooled)
            
            # Compute probability for Class 1 (Real)
            prob = torch.softmax(logits, dim=1)[0, 1].item()
            
            label = row['label']
            source = row['model_name'] # Source algorithm (e.g., MelGAN, WaveGAN)
            
            all_scores.append(prob)
            all_labels.append(label)
            
            # Step 3: Attribution tracking
            if source not in source_results:
                source_results[source] = {"total": 0, "correct": 0}
            
            pred = 1 if prob > 0.5 else 0
            source_results[source]["total"] += 1
            if pred == label:
                source_results[source]["correct"] += 1
                
        except Exception as e:
            # Skip corrupted samples if any
            continue

# --- Final Metric Computation ---
all_labels = np.array(all_labels)
all_scores = np.array(all_scores)

# Accuracy Calculation
final_acc = accuracy_score(all_labels, all_scores > 0.5)

# Equal Error Rate (EER) Calculation
fpr, tpr, thresholds = roc_curve(all_labels, all_scores, pos_label=1)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

# Area Under the ROC Curve (AUC)
auc_score = roc_auc_score(all_labels, all_scores)

# --- Final Performance Report ---
print("\n" + "="*60)
print("🚀 FINQUEST 2026: COMPREHENSIVE TEST ANALYSIS REPORT")
print("="*60)
print(f"✨ Overall Accuracy       : {final_acc:.2%}")
print(f"✨ Equal Error Rate (EER) : {eer:.4f}")
print(f"✨ Area Under Curve (AUC) : {auc_score:.4f}")
print("-" * 60)
print(f"{'Synthetic Source':<18} | {'Count':<8} | {'Accuracy':<10}")
print("-" * 60)

for src, res in source_results.items():
    src_acc = res['correct'] / res['total']
    # If the source is 'original', it refers to Real samples
    source_label = "Real Samples" if src == "original" else f"Fake ({src})"
    print(f"{source_label:<18} | {res['total']:<8} | {src_acc:.2%}")

print("-" * 60)

# --- Traceability Sample Display ---
print("\n📋 Random Sample Traceability Check:")
print(f"{'Filename':<18} | {'Pred':<6} | {'Conf.':<8} | {'Actual':<6} | {'Algorithm'}")
print("-" * 60)

# Select a subset for visualization
test_samples = test_df.sample(min(10, len(test_df)))
for _, row in test_samples.iterrows():
    wav, _ = librosa.load(row['path'], sr=16000, mono=True)
    inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        out = backbone(inputs).last_hidden_state
        logits = classifier(pooling(out))
        prob = torch.softmax(logits, dim=1)[0, 1].item()
    
    pred_str = "REAL" if prob > 0.5 else "FAKE"
    actual_str = "REAL" if row['label'] == 1 else "FAKE"
    fname = os.path.basename(row['path'])[:15] + ".."
    
    print(f"{fname:<18} | {pred_str:<6} | {prob:>7.2%} | {actual_str:<6} | {row['model_name']}")

# Clear memory for subsequent tasks
torch.cuda.empty_cache()
print("\n✅ Final evaluation complete.")

In [2]:
# Cell 8: Multi-Scenario Inference & Evaluation (PERFECT ALIGNMENT VERSION)
import os
import torch
import torch.nn as nn
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# ==============================================================================
# 1. Environment & Configuration (MATCHING CELL 3)
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SR = 16000
MAX_LENGTH = 64000  # 🚀 CRITICAL: Must match Cell 3's max_length

BASE_PATH = Path("/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/Fake_or_Real")
MODEL_PATH = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/model/best_deepfake_model.pth"
MODEL_NAME = "facebook/wav2vec2-base"

# ==============================================================================
# 2. Model Architecture (EXACT MATCH)
# ==============================================================================
class AttentivePooling(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )
    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1)
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

class FinQuestDeepfakeModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        self.pooling = AttentivePooling(768)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    def forward(self, input_values):
        outputs = self.wav2vec2(input_values).last_hidden_state
        pooled = self.pooling(outputs) 
        return self.classifier(pooled)

# ==============================================================================
# 3. Loading Weights
# ==============================================================================
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = FinQuestDeepfakeModel(MODEL_NAME)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model.wav2vec2.load_state_dict(checkpoint["backbone"])
model.pooling.load_state_dict(checkpoint["pooling"])
model.classifier.load_state_dict(checkpoint["classifier"])

model.to(DEVICE)
model.eval()
print("✅ Weights and Architecture aligned. Inference mode active.")

# ==============================================================================
# 4. Corrected Inference Function (REPLICATING CELL 3 LOGIC)
# ==============================================================================
def predict_audio(audio_path, model, processor, device):
    try:
        # Step 1: Load (Matches Cell 3)
        waveform, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
        waveform = torch.from_numpy(waveform)
        
        # Step 2: 🚀 THE MISSING PIECE - Manual Padding/Truncation
        # Must strictly follow the training dataset behavior
        if len(waveform) > MAX_LENGTH:
            waveform = waveform[:MAX_LENGTH]
        else:
            padding_needed = MAX_LENGTH - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, padding_needed))
        
        # Step 3: Feature Extraction
        inputs = processor(waveform, sampling_rate=TARGET_SR, return_tensors="pt")
        input_values = inputs.input_values.to(device)
        
        with torch.no_grad():
            logits = model(input_values)
            probs = torch.softmax(logits, dim=-1)
            return probs[0][1].item() # Real Prob
    except Exception as e:
        return 0.5

# ==============================================================================
# 5. Multi-Scenario Execution
# ==============================================================================
SCENARIOS = ["for-2seconds", "for-norm", "for-original", "for-rerecorded"]
summary_report = {}

for scenario in SCENARIOS:
    test_path = BASE_PATH / scenario / "testing"
    if not test_path.exists(): continue
        
    print(f"\n▶️ Analyzing Scenario: [{scenario}]")
    correct_count, total_count = 0, 0
    
    for label in ["real", "fake"]:
        folder = test_path / label
        if not folder.exists(): continue
        
        for f in tqdm(list(folder.glob("*.wav")), desc=f"  {label}", leave=False):
            score = predict_audio(f, model, processor, DEVICE)
            if (score > 0.5) == (label == "real"):
                correct_count += 1
            total_count += 1

            # 在 score = predict_audio(...) 后面加
            if score > 0.4 and score < 0.6:
                print(f"⚠️ Marginal decision for {f.name}: {score:.4f}")
                    
    if total_count > 0:
        acc = (correct_count / total_count) * 100
        summary_report[scenario] = f"{acc:.2f}%"
        print(f"✨ Accuracy for [{scenario}]: {acc:.2f}%")

print("\n" + "="*60 + "\n📊 FINAL REPORT\n" + "-"*60)
for scene, acc in summary_report.items():
    print(f"Scenario: {scene:<15} | Accuracy: {acc}")
print("="*60)

/home/jesse/Projects/MyEnvs/finquest_2026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 68014.00it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Weights and Architecture aligned. Inference mode active.

▶️ Analyzing Scenario: [for-2seconds]


  fake:  11%|█         | 60/544 [00:00<00:03, 144.18it/s] 

⚠️ Marginal decision for file911.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav: 0.5346
⚠️ Marginal decision for file2022.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav: 0.4115


  fake:  19%|█▉        | 105/544 [00:00<00:03, 144.21it/s]

⚠️ Marginal decision for file191.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav: 0.4962


... (184 outputs left out)

⚠️ Marginal decision for recording13271.wav_norm_mono.wav: 0.4311


  fake:  96%|█████████▌| 390/408 [00:02<00:00, 145.08it/s]

⚠️ Marginal decision for recording13416.wav_norm_mono.wav: 0.4930
⚠️ Marginal decision for recording13432.wav_norm_mono.wav: 0.4451


✨ Accuracy for [for-rerecorded]: 80.39%

📊 FINAL REPORT
------------------------------------------------------------
Scenario: for-2seconds    | Accuracy: 75.00%
Scenario: for-norm        | Accuracy: 80.08%
Scenario: for-original    | Accuracy: 83.53%
Scenario: for-rerecorded  | Accuracy: 80.39%
